# Problem 7: Multi-Task Learning for Speech Emotion Recognition

## Learning Robust Features with Auxiliary Tasks (PyTorch)

---

## Problem Statement

**Goal**: Improve emotion classification by **jointly learning** auxiliary tasks (gender, intensity, speaker) that provide additional supervision signals.

### Why Multi-Task Learning?
- **Shared representations** learn more robust features
- **Regularization effect** - auxiliary tasks prevent overfitting
- **Leverages available labels** - RAVDESS has gender, intensity, speaker info

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import glob
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 2. Data Loading with All Labels

In [ ]:
DATA_DIR = Path("data")

EMOTION_MAP = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised",
}
INTENSITY_MAP = {"01": "normal", "02": "strong"}

def parse_ravdess_filename(filepath):
    name = Path(filepath).stem
    parts = name.split("-")
    if len(parts) != 7:
        return None
    actor_id = int(parts[6])
    return {
        "file_path": filepath,
        "emotion": EMOTION_MAP.get(parts[2]),
        "intensity": INTENSITY_MAP.get(parts[3]),
        "actor_id": parts[6],
        "gender": "male" if actor_id % 2 == 1 else "female",
    }

wav_paths = sorted(glob.glob(str(DATA_DIR / "Actor_*" / "*.wav")))
rows = [r for r in (parse_ravdess_filename(p) for p in wav_paths) if r]
df = pd.DataFrame(rows)

print(f"Dataset size: {len(df)}")
print(f"\nLabel distributions:")
print(f"Emotions: {df['emotion'].nunique()}, Genders: {df['gender'].nunique()}")
print(f"Intensities: {df['intensity'].nunique()}, Speakers: {df['actor_id'].nunique()}")

## 3. Feature Extraction

In [ ]:
SAMPLE_RATE = 22050
N_MELS = 64
MAX_TIME_FRAMES = 128

def extract_mel_spectrogram(file_path):
    try:
        y, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=N_MELS)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        if mel_db.shape[1] < MAX_TIME_FRAMES:
            mel_db = np.pad(mel_db, ((0, 0), (0, MAX_TIME_FRAMES - mel_db.shape[1])), mode='constant')
        else:
            mel_db = mel_db[:, :MAX_TIME_FRAMES]
        return mel_db
    except:
        return None

In [ ]:
print("Extracting features...")

X = []
labels = {'emotion': [], 'gender': [], 'intensity': [], 'speaker': []}

for idx, row in tqdm(df.iterrows(), total=len(df)):
    mel = extract_mel_spectrogram(row['file_path'])
    if mel is not None:
        X.append(mel)
        labels['emotion'].append(row['emotion'])
        labels['gender'].append(row['gender'])
        labels['intensity'].append(row['intensity'])
        labels['speaker'].append(row['actor_id'])

X = np.array(X)[:, np.newaxis, :, :]  # (N, 1, H, W)

# Normalize
X_min, X_max = X.min(), X.max()
X_norm = (X - X_min) / (X_max - X_min)

# Encode all labels
encoders = {}
encoded_labels = {}
for task in ['emotion', 'gender', 'intensity', 'speaker']:
    encoders[task] = LabelEncoder()
    encoded_labels[task] = encoders[task].fit_transform(labels[task])
    print(f"{task}: {len(encoders[task].classes_)} classes")

print(f"\nFeatures shape: {X.shape}")

In [ ]:
# Split data
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED, 
                                        stratify=encoded_labels['emotion'])
train_idx, val_idx = train_test_split(train_idx, test_size=0.15, random_state=SEED)

X_train, X_val, X_test = X_norm[train_idx], X_norm[val_idx], X_norm[test_idx]

y_train = {task: encoded_labels[task][train_idx] for task in encoded_labels}
y_val = {task: encoded_labels[task][val_idx] for task in encoded_labels}
y_test = {task: encoded_labels[task][test_idx] for task in encoded_labels}

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

## 4. Multi-Task Dataset

In [ ]:
class MultiTaskDataset(Dataset):
    def __init__(self, X, y_dict):
        self.X = torch.FloatTensor(X)
        self.y_emotion = torch.LongTensor(y_dict['emotion'])
        self.y_gender = torch.LongTensor(y_dict['gender'])
        self.y_intensity = torch.LongTensor(y_dict['intensity'])
        self.y_speaker = torch.LongTensor(y_dict['speaker'])
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return (self.X[idx], self.y_emotion[idx], self.y_gender[idx],
                self.y_intensity[idx], self.y_speaker[idx])

BATCH_SIZE = 32
train_loader = DataLoader(MultiTaskDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(MultiTaskDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(MultiTaskDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

## 5. Multi-Task Model

In [ ]:
class MultiTaskCNN(nn.Module):
    def __init__(self, n_emotions, n_genders, n_intensities, n_speakers):
        super().__init__()
        
        # Shared CNN encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
            
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        
        # Shared dense layer
        self.shared_fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
        )
        
        # Task-specific heads
        self.emotion_head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n_emotions)
        )
        self.gender_head = nn.Sequential(
            nn.Linear(128, 32), nn.ReLU(), nn.Linear(32, n_genders)
        )
        self.intensity_head = nn.Sequential(
            nn.Linear(128, 32), nn.ReLU(), nn.Linear(32, n_intensities)
        )
        self.speaker_head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n_speakers)
        )
    
    def forward(self, x):
        # Shared encoder
        features = self.encoder(x)
        features = features.view(features.size(0), -1)
        shared = self.shared_fc(features)
        
        # Task outputs
        return {
            'emotion': self.emotion_head(shared),
            'gender': self.gender_head(shared),
            'intensity': self.intensity_head(shared),
            'speaker': self.speaker_head(shared),
        }


n_emotions = len(encoders['emotion'].classes_)
n_genders = len(encoders['gender'].classes_)
n_intensities = len(encoders['intensity'].classes_)
n_speakers = len(encoders['speaker'].classes_)

model = MultiTaskCNN(n_emotions, n_genders, n_intensities, n_speakers).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Training

In [ ]:
# Loss weights
LOSS_WEIGHTS = {'emotion': 1.0, 'gender': 0.3, 'intensity': 0.3, 'speaker': 0.2}

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)


def train_epoch(model, loader, criterion, optimizer, device, weights):
    model.train()
    losses = {task: 0 for task in weights}
    correct = {task: 0 for task in weights}
    total = 0
    
    for X, y_em, y_ge, y_in, y_sp in loader:
        X = X.to(device)
        targets = {'emotion': y_em.to(device), 'gender': y_ge.to(device),
                   'intensity': y_in.to(device), 'speaker': y_sp.to(device)}
        
        optimizer.zero_grad()
        outputs = model(X)
        
        total_loss = 0
        for task in weights:
            loss = criterion(outputs[task], targets[task])
            total_loss += weights[task] * loss
            losses[task] += loss.item()
            
            _, pred = outputs[task].max(1)
            correct[task] += pred.eq(targets[task]).sum().item()
        
        total_loss.backward()
        optimizer.step()
        total += X.size(0)
    
    return {task: losses[task] / len(loader) for task in weights}, \
           {task: correct[task] / total for task in weights}


def evaluate(model, loader, criterion, device, weights):
    model.eval()
    losses = {task: 0 for task in weights}
    correct = {task: 0 for task in weights}
    total = 0
    
    with torch.no_grad():
        for X, y_em, y_ge, y_in, y_sp in loader:
            X = X.to(device)
            targets = {'emotion': y_em.to(device), 'gender': y_ge.to(device),
                       'intensity': y_in.to(device), 'speaker': y_sp.to(device)}
            
            outputs = model(X)
            
            for task in weights:
                loss = criterion(outputs[task], targets[task])
                losses[task] += loss.item()
                _, pred = outputs[task].max(1)
                correct[task] += pred.eq(targets[task]).sum().item()
            
            total += X.size(0)
    
    return {task: losses[task] / len(loader) for task in weights}, \
           {task: correct[task] / total for task in weights}

In [ ]:
NUM_EPOCHS = 100
PATIENCE = 20
best_val_acc = 0
patience_counter = 0

history = {f'{split}_{task}': [] for split in ['train', 'val'] 
           for task in ['emotion', 'gender', 'intensity', 'speaker']}

print("Starting multi-task training...")
for epoch in range(NUM_EPOCHS):
    train_losses, train_accs = train_epoch(model, train_loader, criterion, optimizer, device, LOSS_WEIGHTS)
    val_losses, val_accs = evaluate(model, val_loader, criterion, device, LOSS_WEIGHTS)
    
    scheduler.step(val_accs['emotion'])
    
    for task in LOSS_WEIGHTS:
        history[f'train_{task}'].append(train_accs[task])
        history[f'val_{task}'].append(val_accs[task])
    
    if val_accs['emotion'] > best_val_acc:
        best_val_acc = val_accs['emotion']
        patience_counter = 0
        torch.save(model.state_dict(), 'best_multitask_model.pt')
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - "
              f"Emotion: {val_accs['emotion']:.4f}, Gender: {val_accs['gender']:.4f}, "
              f"Intensity: {val_accs['intensity']:.4f}, Speaker: {val_accs['speaker']:.4f}")
    
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(torch.load('best_multitask_model.pt'))
print("\nTraining complete!")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
tasks = ['emotion', 'gender', 'intensity', 'speaker']
colors = ['steelblue', 'coral', 'seagreen', 'purple']

for ax, task, color in zip(axes.flatten(), tasks, colors):
    ax.plot(history[f'train_{task}'], label='Train', color=color)
    ax.plot(history[f'val_{task}'], label='Validation', color=color, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{task.capitalize()} Task')
    ax.legend()
    ax.grid(True)

plt.suptitle('Multi-Task Training: Accuracy per Task', fontsize=14)
plt.tight_layout()
plt.savefig('multitask_training_curves.png', dpi=150)
plt.show()

## 7. Evaluation

In [ ]:
test_losses, test_accs = evaluate(model, test_loader, criterion, device, LOSS_WEIGHTS)

print("Test Results:")
for task in LOSS_WEIGHTS:
    print(f"  {task.capitalize()} Accuracy: {test_accs[task]:.4f}")

In [ ]:
# Get emotion predictions
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X, y_em, _, _, _ in test_loader:
        X = X.to(device)
        outputs = model(X)
        _, predicted = outputs['emotion'].max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_em.numpy())

print("\nEmotion Classification Report:")
print(classification_report(all_labels, all_preds, target_names=encoders['emotion'].classes_))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=encoders['emotion'].classes_,
            yticklabels=encoders['emotion'].classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Emotion Confusion Matrix (Multi-Task Model)')
plt.tight_layout()
plt.savefig('multitask_emotion_confusion_matrix.png', dpi=150)
plt.show()

## 8. Key Takeaways

### Strengths of Multi-Task Learning
- **Regularization**: Auxiliary tasks prevent overfitting
- **Robust features**: Shared encoder learns generalizable representations
- **Leverages metadata**: Uses all available labels in RAVDESS

### Limitations
- **Task interference**: Conflicting gradients can hurt performance
- **Weight tuning**: Loss weights require careful balancing

### When to Use Multi-Task Learning
- You have **multiple related labels** available
- Your primary task has **limited data**
- You need **multiple predictions**